# FinSight AI: held-out eval on a free Kaggle GPU

Runs `finsight eval` on the held-out question set (32 questions on the Lindsay Corporation FY2025 10-K, including a simulated scanned page)
for four model configs, then writes one results json per config.

Before you start:
- Notebook settings: Accelerator **GPU T4 x2** (or x1), Internet **on**.
- No Hugging Face token is needed, all models are public.
- Nothing here has been run by the author. The vLLM step in particular may need adjusting on Kaggle (see its cell).

Configs:
1. Qwen2.5-VL-3B (fp16, page images)
2. Qwen2.5-VL-7B-Instruct-AWQ served by vLLM (page images)
3. Qwen2-VL-7B-Instruct 4-bit, the original baseline, BM25 page retrieval
4. Qwen2-VL-7B-Instruct 4-bit with ColPali v1.2 retrieval fused with BM25

Run the cells in order. Each eval runs in its own process so GPU memory is freed between configs.

In [ ]:
REF = "v0.2.0"   # git tag or branch of the FinSight-AI repo to test
!git clone --depth 1 --branch $REF https://github.com/mfh-001/FinSight-AI.git
%cd FinSight-AI
!pip install -q -e ".[gpu,colpali]"
!nvidia-smi --query-gpu=name,memory.total --format=csv

The install pins torch 2.7.1 and transformers 4.53.3. If Kaggle asks for a restart after the install, restart the session and run the cells below again, skipping the clone.

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path

RESULTS = Path("results"); RESULTS.mkdir(exist_ok=True)

def run_eval(key, label, backend, model, extra_env=None, quant="none"):
    env = dict(os.environ, FINSIGHT_DEVICE="cuda", FINSIGHT_QUANTIZATION=quant, **(extra_env or {}))
    cmd = ["finsight", "--backend", backend, "--model", model, "eval",
           "--set", "eval/heldout/questions.jsonl", "--docs", "eval/heldout/docs",
           "--label", label, "--out", str(RESULTS / f"{key}.json")]
    t = time.time()
    subprocess.run(cmd, env=env, check=False)
    print(f"{key}: {time.time()-t:.0f} s")
    f = RESULTS / f"{key}.json"
    if f.exists():
        print(json.dumps(json.loads(f.read_text())["summary"], indent=2))
    else:
        print("no result file, check the log above")

## 1. Qwen2.5-VL-3B

In [ ]:
run_eval("qwen25vl-3b", "Qwen2.5-VL-3B, page images", "transformers", "Qwen/Qwen2.5-VL-3B-Instruct",
         {"FINSIGHT_USE_VISION": "true", "FINSIGHT_TOP_K": "4"})

## 3. Qwen2-VL-7B 4-bit, original baseline, BM25 pages

In [ ]:
run_eval("qwen2vl-7b-bm25", "Qwen2-VL-7B 4-bit, BM25 pages", "transformers", "Qwen/Qwen2-VL-7B-Instruct",
         {"FINSIGHT_USE_VISION": "true"}, quant="4bit")

## 4. Qwen2-VL-7B 4-bit with ColPali v1.2 plus BM25

In [ ]:
run_eval("qwen2vl-7b-colpali", "Qwen2-VL-7B 4-bit, ColPali v1.2 + BM25", "transformers", "Qwen/Qwen2-VL-7B-Instruct",
         {"FINSIGHT_USE_VISION": "true", "FINSIGHT_VISUAL_RETRIEVER": "vidore/colpali-v1.2"}, quant="4bit")

## 2. Qwen2.5-VL-7B-Instruct-AWQ with vLLM

vLLM brings its own torch, so it goes in a separate virtualenv. The author could not test this cell. On a T4 (compute capability 7.5)
vLLM may need `VLLM_USE_V1=0` or a different version. If the server does not come up, read `vllm.log`.
If you cannot get it working, leave this config as "not measured yet" and send the other three.

In [ ]:
!python -m venv /kaggle/working/vllm-env
!/kaggle/working/vllm-env/bin/pip install -q vllm==0.10.2

In [ ]:
import urllib.request
log = open("vllm.log", "w")
server = subprocess.Popen(
    ["/kaggle/working/vllm-env/bin/vllm", "serve", "Qwen/Qwen2.5-VL-7B-Instruct-AWQ",
     "--dtype", "half", "--max-model-len", "8192", "--limit-mm-per-prompt", "image=2", "--port", "8000"],
    stdout=log, stderr=subprocess.STDOUT, env=dict(os.environ, VLLM_USE_V1="0"))
for _ in range(120):
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=2); print("server is up"); break
    except Exception:
        time.sleep(10)
else:
    print("server did not start, see vllm.log")

In [ ]:
run_eval("qwen25vl-7b-awq", "Qwen2.5-VL-7B AWQ via vLLM, page images", "openai", "Qwen/Qwen2.5-VL-7B-Instruct-AWQ",
         {"FINSIGHT_LLM_BASE_URL": "http://localhost:8000/v1", "FINSIGHT_USE_VISION": "true"})
server.terminate()

## Collect the results

Download `finsight-results.zip` from the notebook output and send it back. It holds one json per config.

In [ ]:
import shutil
print(sorted(p.name for p in RESULTS.glob("*.json")))
shutil.make_archive("/kaggle/working/finsight-results", "zip", "results")

Back on your machine, unzip into `eval/heldout/results/` and run:

```bash
python scripts/results_to_table.py --update-readme
```

That rewrites the held-out table in the README from the json files. Configs with no file stay "not measured yet".